In [ ]:
import pandas as pd
import re
import numpy as np

### pliki wejściowe
input_files = [
    "gallica_articles.csv", 
    "gallica_articles_2turn.csv", 
    "gallica_articles_3turn.csv"
]
output_file = "gallica_articles_fixed.csv"

def clean_gallica_text(text):
    """
    1. Spłaszcza wiersze / tekst (usuwa entery).
    2. Usuwa tagi <...> (w tym <inc.>, <exposant>).
    3. Usuwa frazę 'Avec mode texte'.
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Spłaszczanie
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Usuwanie tagów (zostawia treść wewnątrz, usuwa same klamry i nazwę tagu)
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Usuwanie specyficznej frazy Gallica
    text = text.replace("Avec mode texte", "")
    
    return text.strip()

def extract_year(val):
    """
    Wyciąga 4 cyfry (rok). 
    Ignoruje '18..' (zwraca NaN), żeby zachować typ liczbowy kolumny.
    """
    if pd.isna(val):
        return np.nan
    val = str(val)
    # szuka ciągu 4 cyfr (od 1000 do 2029)
    match = re.search(r'\b(1\d{3}|20[0-2]\d)\b', val)
    if match:
        return int(match.group(1))
    return np.nan

print("1. Wczytywanie i łączenie plików...")
try:
    ### 1. Wczytuje wszystkie pliki do listy i łączy
    df_list = [pd.read_csv(f, on_bad_lines='skip') for f in input_files]
    df = pd.concat(df_list, ignore_index=True)
    print(f"Połączono. Łącznie wierszy na start: {len(df):,}")
except Exception as e:
    print(f"Błąd krytyczny: {e}")
    df = pd.DataFrame()

if not df.empty:
    ### 2. Czyszczenie tekstów
    print("2. Czyszczenie tekstów (usuwanie tagów i 'Avec mode texte')...")
    text_cols = ['title', 'abstract', 'creator', 'subject', 'publisher']
    
    for col in text_cols:
        if col in df.columns:
            # zamiana nan na pusty string, czyszczenie
            df[col] = df[col].fillna("").astype(str).apply(clean_gallica_text)
            # przywracanie NaN tam gdzie jest pusto (dla porządku)
            df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)

    ### 3. Naprawa dat (obsługa "18..")
    if 'date' in df.columns:
        print("3. Formatowanie daty (zamiana '18..' na NaN, reszta na Int64)...")
        df['year'] = df['date'].apply(extract_year).astype('Int64')
        # usuwanie starej kolumny 'date', zostaje nowa 'year'
        df = df.drop(columns=['date'])

    ### 4. Deduplikacja
    dedup_cols = ['title', 'abstract']
    print(f"4. Deduplikacja wg: {dedup_cols}...")
    initial_count = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    
    print(f"Usunięto duplikatów: {initial_count - len(df):,}")
    print(f"Liczba unikalnych rekordów: {len(df):,}")

    ### UWAGA: Tym razem przy dataset Gallica nie usuwamy wierszy bez abstraktu!

    ### 5. Zapis
    print(f"5. Zapisywanie do {output_file}...")
    cols = [c for c in df.columns if c != 'year']
    if 'year' in df.columns:
        cols.insert(4, 'year') # wstawia rok w sensowne miejsce (zamiast date)

    # zapisanie w ustalonej kolejności
    df[cols].to_csv(output_file, index=False, encoding='utf-8')
    
    # podgląd
    print("\n--- Wynik ---")
    print(df[['title', 'year']].head(5))
    print(f"Abstrakt po czyszczeniu (przykład):")
    # szuka pierwszego niepustego abstraktu
    sample = df['abstract'].dropna().iloc[0] if not df['abstract'].dropna().empty else "Brak abstraktów"
    print(f"'{sample[:100]}...'")
